# Setup

## Libraries

In [ ]:
!pip install -q "pip==24.0"
!pip install \
    "pytorch-lightning==1.7.7" \
    "torchmetrics==0.11.4" \
    "audio-diffusion-pytorch==0.0.43" \
    "audio-data-pytorch==0.0.16" \
    "descript-audiotools>=0.7.2" \
    "einops==0.5.0" \
    "omegaconf" \
    "auraloss" \
    "pyloudnorm" \
    "librosa" \
    "tqdm" \
    "soundfile" \
    "av"
!pip uninstall wandb -y
print("Install complete")

## Clone repo and setup paths

In [ ]:
!git clone https://github.com/XZWY/MSLDM

In [ ]:
import sys

REPO_ROOT = "MSLDM"

sys.path.insert(0, f'{REPO_ROOT}/msldm')
sys.path.insert(0, f'{REPO_ROOT}/SourceVAE')

## Checkpoints

You need two checkpoint files downloaded from Box:

| File             | Download URL                                            |
|------------------|---------------------------------------------------------|
| `sourcevae_ckpt` | https://uofi.box.com/s/as0yxoua68f5dcathvs8yi34far7k705 |
| `msldm.ckpt`     | https://uofi.box.com/s/z2qxbdsxravhdg1n95khz8um3olgeya3 |

It can't be downloaded via wget, so need to manually download and place in the `ckpt` folder. The code below checks that they are present.

In [ ]:
import os

required = ['sourcevae_ckpt', 'msldm.ckpt']
all_ok = True
for f in required:
    path = f'{REPO_ROOT}/ckpt/{f}'
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    status = f'OK ({size_mb:.0f} MB)' if exists else 'MISSING'
    print(f'{f}: {status}')
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError("One or more checkpoints are missing")
print("All checkpoints found")

## Imports and device

In [ ]:
import torch
import torchaudio
import numpy as np
from typing import List, Optional, Callable
from tqdm import tqdm
from IPython.display import Audio, display
import soundfile as sf

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# wandb mock for version mismatch
import sys, types, importlib.machinery

for k in [k for k in sys.modules if "wandb" in k]: del sys.modules[k]

def _m(n): m = types.ModuleType(n); sys.modules[n] = m; return m
w, lib, wr = _m("wandb"), _m("wandb.sdk.lib"), _m("wandb.wandb_run")
sdk = _m("wandb.sdk"); w.sdk = sdk; sdk.lib = lib
w.__spec__ = importlib.machinery.ModuleSpec("wandb", None)

class RunDisabled: pass
class Run: pass
lib.RunDisabled = RunDisabled; wr.Run = Run
w.run = None; w.init = w.log = w.finish = lambda *a, **k: None; w.config = {}


In [ ]:
from audio_diffusion_pytorch import AudioDiffusionModel, KarrasSchedule
from main.module_base_latent import Model
from models.model.dac_vae import DACVAE

In [ ]:
# monkey-patch to force weights_only to be false for older checkpoints
import torch.serialization as _ts

_real_load = _ts.load
torch.load = lambda *a, **kw: _real_load(*a, **{**kw, "weights_only": False})

# Load models

In [ ]:
model = Model.load_from_checkpoint(
    f'{REPO_ROOT}/ckpt/msldm.ckpt',
    # f'{REPO_ROOT}/ckpt/msldm_large.ckpt',
    map_location=device
)
model = model.to(device)
model.eval()
denoise_fn = model.model.diffusion.denoise_fn
print("Diffusion model loaded")

In [ ]:
vae = DACVAE(
    encoder_dim=64,
    encoder_rates=[2, 4, 5, 8],
    latent_dim=80,
    decoder_dim=1536,
    decoder_rates=[8, 5, 4, 2],
    sample_rate=22050
).to(device)

vae_ckpt = torch.load(f'{REPO_ROOT}/ckpt/sourcevae.ckpt', map_location=device, weights_only=False)
vae.load_state_dict(vae_ckpt['generator'])
vae.eval()
print("SourceVAE loaded")

## Helper functions

In [ ]:
STEMS = ['bass', 'drums', 'guitar', 'piano']

def score_differential(x, sigma, denoise_fn):
    d = (x - denoise_fn(x, sigma=sigma)) / sigma
    return d


@torch.no_grad()
def generate_track(
    denoise_fn: Callable,
    sigmas: torch.Tensor,
    noises: torch.Tensor,
    source: Optional[torch.Tensor] = None,
    mask: Optional[torch.Tensor] = None,
    num_resamples: int = 1,
    s_churn: float = 0.0,
    differential_fn: Callable = score_differential,
) -> torch.Tensor:
    x = sigmas[0] * noises
    _, num_sources, _ = x.shape

    source = torch.zeros_like(x) if source is None else source
    mask   = torch.zeros_like(x) if mask   is None else mask

    sigmas = sigmas.to(x.device)
    gamma  = min(s_churn / (len(sigmas) - 1), 2**0.5 - 1)

    for i in tqdm(range(len(sigmas) - 1)):
        sigma, sigma_next = sigmas[i], sigmas[i + 1]

        noisy_source = source + sigma * torch.randn_like(source)

        for r in range(num_resamples):
            x = mask * noisy_source + (1.0 - mask) * x

            sigma_hat = sigma * (gamma + 1)
            x_hat = x + torch.randn_like(x) * (sigma_hat**2 - sigma**2)**0.5

            d = differential_fn(x=x_hat, sigma=sigma_hat, denoise_fn=denoise_fn)
            x = x_hat + d * (sigma_next - sigma_hat)

            if r < num_resamples - 1:
                x = x + torch.randn_like(x) * (sigma**2 - sigma_next**2)**0.5

    return mask * source + (1.0 - mask) * x


print("generate_track defined")

import os
# Total Generation

In [ ]:
num_steps = 150
s_churn = 20.0
batch_size = 1
latent_dim = 80

schedule = KarrasSchedule(sigma_min=1e-2, sigma_max=3, rho=7)(num_steps, device)

print(f"Running {num_steps} diffusion steps")
with torch.no_grad():
    generated = generate_track(
        denoise_fn,
        sigmas=schedule,
        noises=torch.randn(batch_size, latent_dim * 4, 1024).to(device),
        s_churn=s_churn,
        num_resamples=1,
    )

# Decode latents -> waveform via SourceVAE
generated_flat = generated.reshape(batch_size * 4, latent_dim, -1)
with torch.no_grad():
    waves_gen = vae.decode(generated_flat)          # (B*4, 1, T)
waves_gen = waves_gen.reshape(batch_size, 4, -1)    # (B, 4, T)
waves_gen = waves_gen.squeeze(0).cpu().numpy()      # (4, T)

print(f"Done. Shape: {waves_gen.shape}  |  Duration: {waves_gen.shape[-1]/22050:.1f}s at 22050 Hz")

In [ ]:
# Listen to the generated stems
for i, name in enumerate(STEMS):
    print(f"{name}:")
    display(Audio(waves_gen[i], rate=22050))

print("Mixture (all stems summed):")
display(Audio(waves_gen.sum(0), rate=22050))

In [ ]:
# Save generated stems to /content/generated/
OUT_DIR = '/content/generated'
os.makedirs(OUT_DIR, exist_ok=True)

for i, name in enumerate(STEMS):
    sf.write(f'{OUT_DIR}/{name}.wav', waves_gen[i], 22050)

sf.write(f'{OUT_DIR}/mixture.wav', waves_gen.sum(0), 22050)
print(f"Saved to {OUT_DIR}:", os.listdir(OUT_DIR))

# Batch Generation

In [ ]:
NUM_SAMPLES = 16      # total number of tracks to generate
BATCH_SIZE  = 4       # tracks per diffusion forward pass
NUM_STEPS   = 150
S_CHURN     = 20.0
LATENT_DIM  = 80
SR          = 22050
OUT_DIR     = "generated"

os.makedirs(OUT_DIR, exist_ok=True)

schedule = KarrasSchedule(sigma_min=1e-2, sigma_max=3, rho=7)(NUM_STEPS, device)
num_batches = (NUM_SAMPLES + BATCH_SIZE - 1) // BATCH_SIZE
generated_count = 0

for batch_idx in range(num_batches):
    current_batch = min(BATCH_SIZE, NUM_SAMPLES - generated_count)
    print()
    print(f"Batch {batch_idx + 1}/{num_batches}  ({current_batch} tracks)")

    with torch.no_grad():
        latents = generate_track(
            denoise_fn,
            sigmas=schedule,
            noises=torch.randn(current_batch, LATENT_DIM * 4, 1024).to(device),
            s_churn=S_CHURN,
            num_resamples=1,
        )

    # Decode latents -> waveforms
    flat = latents.reshape(current_batch * 4, LATENT_DIM, -1)
    with torch.no_grad():
        waves = vae.decode(flat)                              # (B*4, 1, T)
    waves = waves.reshape(current_batch, 4, -1).cpu().numpy()  # (B, 4, T)

    for i in range(current_batch):
        sample_idx = generated_count + i
        stem_waves = waves[i]       # (4, T)
        mixture    = stem_waves.sum(0)

        for j, name in enumerate(STEMS):
            sf.write(f"{OUT_DIR}/{sample_idx:04d}_{name}.wav", stem_waves[j], SR)
        sf.write(f"{OUT_DIR}/{sample_idx:04d}_mixture.wav", mixture, SR)

    generated_count += current_batch
    lo, hi = generated_count - current_batch, generated_count - 1
    print(f"  Saved {lo:04d}–{hi:04d} -> {OUT_DIR}/")

print()
print(f"Done. {generated_count} tracks saved to {OUT_DIR}/")
